# 🔶 Bronze Layer - Análise Exploratória com PySpark

## 📋 Objetivo

Perfilamento escalável dos dados brutos de Dengue e validação de schema para preparar transformação Silver Layer.
---

## 📊 Dataset

- **Fonte**: SINAN - Sistema de Informação de Agravos de Notificação
- **Doença**: Dengue (CID A90)
- **Período**: 2024-2026
- **Registros**: ~1.6M notificações
- **Variáveis**: 121 colunas

---

## 🔄 Fluxo de Análise

1. **Amostragem (Top 100)**: Calibração rápida
2. **Full Load Spark**: Carregamento completo
3. **Data Quality**: Missing values e completude
4. **Validação de Idade**: Decodificar códigos SINAN
5. **Validação Temporal**: Datas e consistência
6. **Validação Geográfica**: UF e municípios
7. **Validação Clínica**: Sintomas e evolução
8. **Output**: Plano de transformação Silver

---

# 🔧 Setup e Configuração

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from datetime import datetime

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, count, when, length, to_date, 
    year, month, day, udf, desc, asc
)
from pyspark.sql.types import IntegerType, FloatType, StringType

import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

In [ ]:
DATA_PATH = 'data/DENGBR25.csv'
OUTPUT_SUMMARY = 'bronze_spark_summary.json'
OUTPUT_PLAN = 'silver_transformation_plan.md'
OUTPUT_REPORT = 'data_quality_report.md'

---

# 📦 Etapa 1: Amostragem Inicial (Top 100)

**Objetivo**: Descobrir encoding, separador e estrutura antes de carregar com Spark.

Usamos Pandas para leitura leve e rápida de apenas 100 linhas.

In [ ]:
sample_df = pd.read_csv(
    DATA_PATH,
    nrows=100,
    encoding='latin-1',
    sep=None,
    engine='python'
)

print(f"✅ Amostra carregada: {sample_df.shape[0]} linhas x {sample_df.shape[1]} colunas")
print(f"\nEncoding detectado: latin-1")
print(f"Separador: vírgula (,)")
print(f"\nPrimeiras colunas: {sample_df.columns[:10].tolist()}")

In [ ]:
sample_df.head(5)

---

# ⚡ Etapa 2: Inicialização Spark Session

**Configuração**:
- `local[*]`: Usa todos os núcleos da máquina
- `4g`: Memória do driver
- `8 partições`: Otimizado para dataset de ~1.6M registros

In [ ]:
spark = SparkSession.builder \
    .appName("Bronze_Dengue_Analysis") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

print(f"✅ Spark Session criada")
print(f"Versão: {spark.version}")
print(f"Master: {spark.sparkContext.master}")

---

# 📥 Etapa 3: Full Load com Spark

**Ação**: Carregar dataset completo com schema inferido.

Spark irá:
1. Ler todo o CSV
2. Inferir tipos de dados
3. Distribuir dados em partições

In [ ]:
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("encoding", "latin1") \
    .csv(DATA_PATH)

total_records = df.count()
total_columns = len(df.columns)

print(f"✅ Dataset carregado com sucesso!")
print(f"\nRegistros: {total_records:,}")
print(f"Colunas: {total_columns}")
print(f"Total de células: {total_records * total_columns:,}")

In [ ]:
df.printSchema()

---

# 🔍 Etapa 4: Data Quality - Missing Values

**Análise de Completude**:
- Contar nulos por coluna usando Spark SQL
- Calcular percentuais
- Identificar colunas 100% vazias (candidatas a remoção)

In [ ]:
missing_counts = df.select([
    count(when(col(c).isNull(), c)).alias(c) 
    for c in df.columns
]).collect()[0].asDict()

missing_data = []
for col_name, null_count in missing_counts.items():
    pct = (null_count / total_records) * 100
    missing_data.append({
        'coluna': col_name,
        'nulos': null_count,
        'percentual': round(pct, 2)
    })

missing_df = pd.DataFrame(missing_data).sort_values('percentual', ascending=False)

print(f"📊 Análise de Missing Values")
print(f"\nColunas com dados ausentes: {len(missing_df[missing_df['nulos'] > 0])}")
print(f"Colunas 100% vazias: {len(missing_df[missing_df['percentual'] == 100])}")

In [ ]:
print("\nTop 20 colunas com mais dados ausentes:\n")
print(missing_df.head(20).to_string(index=False))

In [ ]:
empty_columns = missing_df[missing_df['percentual'] == 100]['coluna'].tolist()
print(f"\n🗑️ Colunas 100% vazias ({len(empty_columns)}):")
print(empty_columns)

### 💡 Insight: Completude

Colunas 100% vazias devem ser removidas na camada Silver para otimizar storage e performance.

---

# 🎂 Etapa 5: Validação de Idade

**Problema**: Campo `NU_IDADE_N` usa códigos SINAN:
- `1000-2999`: Idade em **dias**
- `3000-3999`: Idade em **meses**
- `4000+`: Idade em **anos** (código - 4000)

**Solução**: Criar UDF para decodificar idade real em anos.

In [ ]:
@udf(returnType=FloatType())
def decode_age(age_code):
    if age_code is None:
        return None
    
    if age_code >= 4000:
        return float(age_code - 4000)
    elif age_code >= 3000:
        return (age_code - 3000) / 12.0
    else:
        return (age_code - 1000) / 365.0

@udf(returnType=IntegerType())
def get_age_unit(age_code):
    if age_code is None:
        return None
    if age_code >= 4000:
        return 3
    elif age_code >= 3000:
        return 2
    else:
        return 1

In [ ]:
df_with_age = df.withColumn("IDADE_ANOS", decode_age(col("NU_IDADE_N"))) \
                .withColumn("IDADE_UNIDADE", get_age_unit(col("NU_IDADE_N")))

print("✅ Idade decodificada com sucesso")
print("\nAmostra de decodificação:")
df_with_age.select("NU_IDADE_N", "IDADE_ANOS", "IDADE_UNIDADE").show(10)

In [ ]:
age_distribution = df_with_age.groupBy("IDADE_UNIDADE") \
    .count() \
    .orderBy("IDADE_UNIDADE") \
    .toPandas()

age_distribution['unidade_nome'] = age_distribution['IDADE_UNIDADE'].map({
    1: 'Dias',
    2: 'Meses',
    3: 'Anos'
})

print("\n📊 Distribuição por unidade de idade:")
print(age_distribution[['unidade_nome', 'count']])

### 💡 Insight: Idade

UDF `decode_age` e `get_age_unit` devem ser aplicadas na camada Silver para criar colunas `IDADE_ANOS` (Float) e `IDADE_UNIDADE` (Integer).

---

# 📅 Etapa 6: Validação Temporal

**Validações**:
1. Formato de datas
2. DT_SIN_PRI (sintomas) ≤ DT_NOTIFIC (notificação)
3. Sem datas futuras
4. Ano no range válido

In [ ]:
df.createOrReplaceTempView("dengue_raw")

In [ ]:
date_analysis = spark.sql("""
    SELECT 
        NU_ANO,
        COUNT(*) as casos
    FROM dengue_raw
    GROUP BY NU_ANO
    ORDER BY NU_ANO
""")

print("📅 Distribuição por ano:")
date_analysis.show()

In [ ]:
date_format_check = spark.sql("""
    SELECT 
        DT_NOTIFIC,
        DT_SIN_PRI,
        COUNT(*) as casos
    FROM dengue_raw
    GROUP BY DT_NOTIFIC, DT_SIN_PRI
    ORDER BY casos DESC
    LIMIT 10
""")

print("\n📋 Amostra de formatos de data:")
date_format_check.show()

### 💡 Insight: Temporal

Datas estão no formato `YYYY-MM-DD`. Na Silver:
- Cast para `DateType`
- Validar DT_SIN_PRI ≤ DT_NOTIFIC
- Filtrar anos inválidos (2024, 2026 suspeitos)

---

# 🗺️ Etapa 7: Validação Geográfica

**Validações**:
1. Distribuição por UF
2. Tamanho do código de município (deve ser 7 dígitos)
3. Consistência UF x Município

In [ ]:
uf_distribution = spark.sql("""
    SELECT 
        SG_UF_NOT,
        COUNT(*) as casos
    FROM dengue_raw
    GROUP BY SG_UF_NOT
    ORDER BY casos DESC
""")

print("🗺️ Top 10 UFs com mais notificações:")
uf_distribution.show(10)

In [ ]:
municipio_length_check = df.withColumn(
    "codigo_length",
    length(col("ID_MUNICIP").cast(StringType()))
).groupBy("codigo_length").count().orderBy("codigo_length")

print("\n📏 Distribuição por tamanho de código de município:")
municipio_length_check.show()

In [ ]:
@udf(returnType=StringType())
def standardize_municipio_codigo(codigo):
    if codigo is None:
        return None
    return str(int(codigo)).zfill(7)

df_with_municipio = df.withColumn(
    "ID_MUNICIP_STD",
    standardize_municipio_codigo(col("ID_MUNICIP"))
)

print("✅ Códigos de município padronizados")
df_with_municipio.select("ID_MUNICIP", "ID_MUNICIP_STD").show(5)

### 💡 Insight: Geografia

Na Silver:
- Aplicar UDF `standardize_municipio_codigo` para padronizar códigos
- Todos os municípios devem ter 7 dígitos
- Validar consistência: primeiros 2 dígitos = código UF

---

# 🏥 Etapa 8: Validação Clínica

**Análise de**:
- Sintomas (FEBRE, CEFALEIA, etc.)
- Classificação final
- Hospitalização
- Evolução dos casos

In [ ]:
symptom_columns = ['FEBRE', 'MIALGIA', 'CEFALEIA', 'EXANTEMA', 'VOMITO', 'NAUSEA']

for symptom in symptom_columns:
    if symptom in df.columns:
        distinct_values = df.select(symptom).distinct().collect()
        values = [row[0] for row in distinct_values if row[0] is not None]
        print(f"{symptom}: {sorted(values)}")

In [ ]:
classi_final = spark.sql("""
    SELECT 
        CLASSI_FIN,
        COUNT(*) as casos
    FROM dengue_raw
    WHERE CLASSI_FIN IS NOT NULL
    GROUP BY CLASSI_FIN
    ORDER BY casos DESC
""")

print("\n🏥 Classificação Final:")
classi_final.show()

In [ ]:
hospitalizacao = spark.sql("""
    SELECT 
        HOSPITALIZ,
        COUNT(*) as casos
    FROM dengue_raw
    WHERE HOSPITALIZ IS NOT NULL
    GROUP BY HOSPITALIZ
    ORDER BY casos DESC
""")

print("\n🏨 Hospitalização:")
hospitalizacao.show()

In [ ]:
evolucao = spark.sql("""
    SELECT 
        EVOLUCAO,
        COUNT(*) as casos
    FROM dengue_raw
    WHERE EVOLUCAO IS NOT NULL
    GROUP BY EVOLUCAO
    ORDER BY casos DESC
""")

print("\n💉 Evolução dos Casos:")
evolucao.show()

### 💡 Insight: Clínica

Sintomas usam códigos:
- **1**: Sim
- **2**: Não
- **NULL**: Ignorado

Na Silver: Cast para IntegerType e criar dicionário de decodificação.

---

# 📊 Etapa 9: Geração de Outputs

## Output 1: Resumo JSON

In [ ]:
summary = {
    "total_notificacoes": total_records,
    "total_variaveis": total_columns,
    "periodo_analise": "2024-2026",
    "ufs_cobertas": df.select("SG_UF_NOT").distinct().count(),
    "colunas_100_vazias": len(empty_columns),
    "percentual_missing_geral": round(
        sum(missing_df['nulos']) / (total_records * total_columns) * 100, 2
    ),
    "data_execucao": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
}

with open(OUTPUT_SUMMARY, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(f"✅ Resumo salvo em: {OUTPUT_SUMMARY}")
print(json.dumps(summary, indent=2, ensure_ascii=False))

## Output 2: Plano de Transformação Silver

In [ ]:
silver_plan = f"""# Plano de Transformação: Silver Layer

**Gerado automaticamente**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

---

## 1. Limpeza de Dados

### Remover Colunas 100% Vazias ({len(empty_columns)} colunas)

```python
columns_to_drop = {empty_columns}
df_silver = df_bronze.drop(*columns_to_drop)
```

---

## 2. UDFs Necessárias

### UDF: Decodificar Idade

```python
@udf(returnType=FloatType())
def decode_age(age_code):
    if age_code is None:
        return None
    if age_code >= 4000:
        return float(age_code - 4000)
    elif age_code >= 3000:
        return (age_code - 3000) / 12.0
    else:
        return (age_code - 1000) / 365.0

df_silver = df_silver.withColumn("IDADE_ANOS", decode_age(col("NU_IDADE_N")))
```

### UDF: Padronizar Município

```python
@udf(returnType=StringType())
def standardize_municipio_codigo(codigo):
    if codigo is None:
        return None
    return str(int(codigo)).zfill(7)

df_silver = df_silver.withColumn(
    "ID_MUNICIP_STD",
    standardize_municipio_codigo(col("ID_MUNICIP"))
)
```

---

## 3. Schema Enforcement

```python
from pyspark.sql.types import StructType, StructField, DateType, IntegerType, StringType

schema_silver = StructType([
    StructField("DT_NOTIFIC", DateType(), False),
    StructField("NU_ANO", IntegerType(), False),
    StructField("SG_UF_NOT", StringType(), False),
    StructField("ID_MUNICIP_STD", StringType(), True),
    StructField("IDADE_ANOS", FloatType(), True),
    # ... adicionar outros campos
])
```

---

## 4. Transformações de Colunas

### Datas para DateType

```python
df_silver = df_silver \
    .withColumn("DT_NOTIFIC", to_date(col("DT_NOTIFIC"), "yyyy-MM-dd")) \
    .withColumn("DT_SIN_PRI", to_date(col("DT_SIN_PRI"), "yyyy-MM-dd"))
```

### Sintomas para IntegerType

```python
symptom_cols = ['FEBRE', 'MIALGIA', 'CEFALEIA', 'EXANTEMA', 'VOMITO', 'NAUSEA']

for symptom in symptom_cols:
    df_silver = df_silver.withColumn(symptom, col(symptom).cast(IntegerType()))
```

---

## 5. Validações de Qualidade

```python
df_silver = df_silver.filter(
    (col("DT_SIN_PRI") <= col("DT_NOTIFIC")) &
    (col("NU_ANO") >= 2020) &
    (col("NU_ANO") <= 2025)
)
```

---

## 6. Otimização: Parquet & Particionamento

```python
df_silver.write \
    .mode("overwrite") \
    .partitionBy("NU_ANO", "SG_UF_NOT") \
    .parquet("data/silver/dengue")
```

**Benefícios**:
- Formato Parquet: 70-90% de economia de espaço vs CSV
- Particionamento: Queries filtradas por ano/UF são 10-100x mais rápidas

---

## 7. Métricas Pós-Transformação

```python
print(f"Registros Bronze: {{df_bronze.count():,}}")
print(f"Registros Silver: {{df_silver.count():,}}")
print(f"Registros removidos: {{df_bronze.count() - df_silver.count():,}}")
```
"""

with open(OUTPUT_PLAN, 'w', encoding='utf-8') as f:
    f.write(silver_plan)

print(f"\n✅ Plano Silver salvo em: {OUTPUT_PLAN}")

## Output 3: Relatório de Data Quality

In [ ]:
quality_report = f"""# Relatório de Data Quality - Bronze Layer

**Data**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

---

## Resumo Executivo

- **Total de Registros**: {total_records:,}
- **Total de Variáveis**: {total_columns}
- **% Missing Geral**: {summary['percentual_missing_geral']}%
- **Colunas 100% Vazias**: {len(empty_columns)}

---

## Problemas Identificados

### ❌ Colunas 100% Vazias

Estas {len(empty_columns)} colunas devem ser removidas:

{chr(10).join(f'- `{col}`' for col in empty_columns[:20])}

### ⚠️ Top 10 Colunas com Mais Missing

{missing_df.head(10)[['coluna', 'percentual']].to_markdown(index=False)}

---

## Recomendações

1. **Remover** colunas 100% vazias
2. **Decodificar** campo NU_IDADE_N
3. **Padronizar** códigos de município (7 dígitos)
4. **Validar** datas (formato e consistência)
5. **Cast** sintomas para IntegerType

---

**Próximo Passo**: Implementar transformações Silver conforme plano.
"""

with open(OUTPUT_REPORT, 'w', encoding='utf-8') as f:
    f.write(quality_report)

print(f"✅ Relatório quality salvo em: {OUTPUT_REPORT}")

---

# 🛑 Encerramento

In [ ]:
spark.stop()
print("✅ Spark Session encerrada")
print("\n" + "="*80)
print("🎉 ANÁLISE BRONZE CONCLUÍDA COM SUCESSO!")
print("="*80)

---

# 📚 Referências e Próximos Passos

## Arquivos Gerados

1. `bronze_spark_summary.json` - Resumo estatístico
2. `silver_transformation_plan.md` - Plano de transformação
3. `data_quality_report.md` - Relatório de qualidade

## Próximos Passos

1. ✅ Análise Bronze concluída
2. 📋 **Próximo**: Implementar ETL Bronze → Silver
3. 🎨 **Depois**: Criar camada Gold (agregações)
4. 📊 **Final**: Dashboards e visualizações

## Documentação

- [PySpark SQL Functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html)
- [SINAN - Dicionário de Dados](http://portalsinan.saude.gov.br/)
- [Data Quality Best Practices](https://www.databricks.com/glossary/data-quality)

---

**Versão**: 1.0  
**Autor**: Arthur Gabrieel  
**Data**: 2026-01-15